# Notebook 00b. Repartición por grupos (misma hoja = misma partición)
Reemplaza la partición del notebook 00. **No vuelve a reducir las imágenes**: usa el mismo `X224.npy` (mismo orden) y solo rehace `meta.csv` y `stats.json`.

**Por qué:** varias fotos son ráfagas de la misma hoja (tomadas con segundos de diferencia). Con la partición 80/10/10 aleatoria del notebook 00, 162 de las 236 imágenes de test tenían en train una foto de la misma clase tomada 20 s o menos antes o después, así que el test no medía generalización a hojas nuevas.

**Qué hace:**
1. Lee la hora de captura del nombre de archivo (`IMG_20250426_083809`, `20250621_112345`). Los nombres de 13 dígitos (`1746918118961`) son la hora de exportación/descarga, no de captura (286 fotos distintas "tomadas" en 2 minutos), así que no se usan como tiempo.
2. Une en un mismo grupo: (a) fotos consecutivas de la misma clase con ≤ 20 s entre sí; (b) pares de la misma clase visualmente casi idénticos según ResNet50-ImageNet (coseno ≥ 0.92); (c) pares de la misma clase con ≥ 20 puntos SIFT coincidentes bajo una homografía (RANSAC).
3. Parte 80/10/10 con `StratifiedGroupKFold` (10 folds, semilla 42): fold 0 = test, fold 1 = val, resto = train. Un grupo nunca queda en dos particiones.
4. Auditoría visual: se revisó cada imagen de test junto a su foto más parecida de train (235 pares). 2 pares parecían la misma hoja; esos grupos pasan a train (celda 5b).
5. Recalcula media, desviación y pesos de clase solo con train.

Settings: Accelerator **None**, Internet **On** (descarga ResNet50 ONNX de GitHub). Input: dataset privado **banana-cache-224** (versión 1). Salida: los 3 archivos del caché nuevo + `grupos_enlaces.csv`.
Versiones con las que se generaron los archivos entregados: opencv 5.0.0, onnxruntime 1.29.0, scikit-learn 1.9.1, numpy 2.5.3. Con otras versiones de opencv los grupos pueden variar mínimamente; en ese caso, los archivos entregados son la referencia.

In [ ]:
# CELDA 1: cargar el caché v1 y leer la hora de captura del nombre
!pip -q install onnxruntime onnx
import os, re, glob, json, shutil, urllib.request, hashlib
from datetime import datetime
import numpy as np, pandas as pd, cv2
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

CACHE = glob.glob("/kaggle/input/**/X224.npy", recursive=True)
assert len(CACHE) == 1, CACHE
DIR, OUT = os.path.dirname(CACHE[0]), "/kaggle/working"
X = np.load(CACHE[0], mmap_mode="r")
meta = pd.read_csv(f"{DIR}/meta.csv")[["file_name", "class_name", "label", "sample_id"]]
assert X.shape == (len(meta), 224, 224, 3) and len(meta) == 2354
CLASES = ["sigatoka", "cordana", "sanas"]
T_RAFAGA, COS_MIN, SIFT_MIN = 20, 0.92, 20   # umbrales fijados

def hora_captura(f):
    s = re.sub(r"\(\d+\)$", "", f.rsplit(".", 1)[0])
    r = re.fullmatch(r"(?:IMG_)?(\d{8})_(\d{6})", s)
    return datetime.strptime(r.group(1) + r.group(2), "%Y%m%d%H%M%S").timestamp() if r else np.nan

meta["t"] = meta.file_name.map(hora_captura)
print("con hora de camara:", meta.t.notna().sum(), "| sin hora de camara:", meta.t.isna().sum())
cls = meta.class_name.to_numpy(dtype=object)

In [ ]:
# CELDA 2: pares candidatos y verificacion SIFT + homografia
def mini(i):
    g = np.asarray(Image.fromarray(np.array(X[i])).convert("L").resize((32, 32), Image.BOX), np.float32).ravel()
    return g - g.mean()
F = np.stack([mini(i) for i in range(len(X))]); F /= np.linalg.norm(F, axis=1, keepdims=True) + 1e-9
Hc = np.stack([np.histogramdd(np.array(X[i]).reshape(-1, 3) // 32, bins=(8, 8, 8), range=((0, 8),) * 3)[0].ravel()
               for i in range(len(X))]).astype(np.float32)
Hc = np.sqrt(Hc / Hc.sum(1, keepdims=True))
cand = set()
for M in (F @ F.T, Hc @ Hc.T, 0.5 * (F @ F.T) + 0.5 * (Hc @ Hc.T)):
    np.fill_diagonal(M, -9)
    for i, row in enumerate(np.argsort(-M, 1)[:, :15]):
        cand.update((min(i, j), max(i, j)) for j in row)
cam, t = meta.t.notna().values, meta.t.values
for c in CLASES:
    ii = np.where((cls == c) & cam)[0]
    cand.update((a, b) for a in ii for b in ii if a < b and abs(t[a] - t[b]) <= 600)
    ii = np.where((cls == c) & ~cam)[0]
    cand.update((a, b) for a in ii for b in ii if a < b)
cand = np.array(sorted(cand))

sift = cv2.SIFT_create(nfeatures=400)
KP = []
for i in range(len(X)):
    kp, d = sift.detectAndCompute(cv2.cvtColor(np.array(X[i]), cv2.COLOR_RGB2GRAY), None)
    KP.append((np.array([k.pt for k in kp], np.float32), d if d is not None else np.zeros((0, 128), np.float32)))
bf = cv2.BFMatcher(cv2.NORM_L2)
def inliers(i, j):
    (pa, da), (pb, db) = KP[i], KP[j]
    if len(da) < 8 or len(db) < 8:
        return 0
    good = [a for a, b in (x for x in bf.knnMatch(da, db, k=2) if len(x) == 2) if a.distance < 0.75 * b.distance]
    if len(good) < 8:
        return len(good) // 2
    H, mask = cv2.findHomography(pa[[g.queryIdx for g in good]], pb[[g.trainIdx for g in good]], cv2.RANSAC, 3.0)
    return int(mask.sum()) if mask is not None else 0
inl = np.array([inliers(a, b) for a, b in cand])
print("pares candidatos:", len(cand), "| con >= %d puntos SIFT:" % SIFT_MIN, (inl >= SIFT_MIN).sum())

In [ ]:
# CELDA 3: similitud visual con ResNet50 preentrenada en ImageNet (ONNX)
import onnx, onnxruntime as ort
URL = "https://media.githubusercontent.com/media/onnx/models/main/validated/vision/classification/resnet/model/resnet50-v1-12.onnx"
if not os.path.exists("resnet50.onnx"):
    urllib.request.urlretrieve(URL, "resnet50.onnx")
assert hashlib.sha256(open("resnet50.onnx", "rb").read()).hexdigest().startswith("3f03fdef724b2294")
g = onnx.load("resnet50.onnx")
g.graph.output.append(onnx.helper.make_tensor_value_info("flatten_473", onnx.TensorProto.FLOAT, None))
sess = ort.InferenceSession(g.SerializeToString(), providers=["CPUExecutionProvider"])
MU = np.array([0.485, 0.456, 0.406], np.float32).reshape(1, 3, 1, 1); SD = np.array([0.229, 0.224, 0.225], np.float32).reshape(1, 3, 1, 1)
E = np.concatenate([sess.run(["flatten_473"], {"data": (np.array(X[i:i + 32]).transpose(0, 3, 1, 2).astype(np.float32) / 255 - MU) / SD})[0]
                    for i in range(0, len(X), 32)])
E /= np.linalg.norm(E, axis=1, keepdims=True)
S = E @ E.T
print("embeddings:", E.shape)

In [ ]:
# CELDA 4: enlaces y grupos (union-find)
enl = []
c = meta[cam].sort_values(["class_name", "t"])
for _, gg in c.groupby("class_name"):
    ii, tt = gg.index.values, gg.t.values
    enl += [(ii[k - 1], ii[k], "rafaga_tiempo", tt[k] - tt[k - 1]) for k in range(1, len(ii)) if tt[k] - tt[k - 1] <= T_RAFAGA]
a, b = np.where(np.triu(S >= COS_MIN, 1))
enl += [(x, y, "visual_resnet50", float(S[x, y])) for x, y in zip(a, b) if cls[x] == cls[y]]
enl += [(x, y, "visual_sift", int(v)) for (x, y), v in zip(cand, inl) if v >= SIFT_MIN and cls[x] == cls[y]]

par = np.arange(len(meta))
def raiz(x):
    while par[x] != x:
        par[x] = par[par[x]]; x = par[x]
    return x
for x, y, *_ in enl:
    rx, ry = raiz(x), raiz(y)
    if rx != ry:
        par[max(rx, ry)] = min(rx, ry)
_, meta["group_id"] = np.unique([raiz(i) for i in range(len(meta))], return_inverse=True)
tam = meta.group_id.value_counts()
print("enlaces:", pd.Series([e[2] for e in enl]).value_counts().to_dict())
print("grupos:", len(tam), "| sueltas:", (tam == 1).sum(), "| grupo mas grande:", tam.max())

In [ ]:
# CELDA 5: particion 80/10/10 estratificada POR GRUPOS, semilla 42
folds = list(StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42).split(meta, meta.label, meta.group_id))
meta["split"] = "train"
meta.loc[folds[0][1], "split"] = "test"
meta.loc[folds[1][1], "split"] = "val"
assert meta.groupby("group_id").split.nunique().max() == 1, "un grupo quedo partido"
print(pd.crosstab(meta.class_name, meta.split, margins=True)[["train", "val", "test", "All"]])
print((pd.crosstab(meta.class_name, meta.split, normalize="columns") * 100).round(1))

In [ ]:
# CELDA 5b: correcciones de la auditoria visual (cada imagen de test junto a su foto mas parecida de train)
# Pares que a ojo parecen la misma hoja y que las reglas automaticas no unieron.
# Se unen en un grupo y el grupo de test pasa a la particion de su pareja (train).
ENLACES_MANUALES = [
    ("cordana/1746918118795.jpg", "cordana/1746918118790.jpg"),             # misma hoja probable (nombres de exportacion seguidos)
    ("cordana/IMG_20250622_095009.jpg", "cordana/IMG_20250622_094839.jpg"),  # misma hoja posible (90 s, lesion casi igual)
]
for a, b in ENLACES_MANUALES:
    ga, gb = meta.group_id[meta.sample_id == a].item(), meta.group_id[meta.sample_id == b].item()
    destino = meta.split[meta.group_id == gb].iat[0]
    meta.loc[meta.group_id == ga, "split"] = destino
    meta.loc[meta.group_id == ga, "group_id"] = gb
    enl.append((meta.index[meta.sample_id == a][0], meta.index[meta.sample_id == b][0], "auditoria_visual", 1))
assert meta.groupby("group_id").split.nunique().max() == 1
_, meta["group_id"] = np.unique(meta.group_id, return_inverse=True)
tam = meta.group_id.value_counts()
print(pd.crosstab(meta.class_name, meta.split, margins=True)[["train", "val", "test", "All"]])

In [ ]:
# CELDA 6: estadisticas SOLO con train (mismo calculo que el notebook 00) y guardar
tr_idx = np.where(meta.split == "train")[0]
s, ss, n = np.zeros(3), np.zeros(3), 0
for i in tr_idx:
    x = X[i].astype(np.float64) / 255.0
    s += x.sum((0, 1)); ss += (x ** 2).sum((0, 1)); n += x.shape[0] * x.shape[1]
mean = s / n
std = np.sqrt(ss / n - mean ** 2)
n_c = np.bincount(meta.label.values[tr_idx], minlength=3)
pesos = len(tr_idx) / (3 * n_c)
stats = {"mean_train": mean.tolist(), "std_train": std.tolist(), "imagenes_train_por_clase": n_c.tolist(),
         "pesos_clase_train": pesos.tolist(), "clases": CLASES,
         "split": "80/10/10 estratificado por grupos (StratifiedGroupKFold), semilla 42",
         "grupos": {"rafaga_s": T_RAFAGA, "coseno_resnet50": COS_MIN, "sift_inliers": SIFT_MIN, "n_grupos": int(len(tam))}}
json.dump(stats, open(f"{OUT}/stats.json", "w"), indent=1)
meta[["file_name", "class_name", "label", "sample_id", "split", "group_id"]].to_csv(f"{OUT}/meta.csv", index=False)
sid = meta.sample_id.values
pd.DataFrame([(sid[x], sid[y], k, v) for x, y, k, v in enl], columns=["sample_a", "sample_b", "motivo", "valor"]).to_csv(f"{OUT}/grupos_enlaces.csv", index=False)
if os.path.abspath(CACHE[0]) != os.path.abspath(f"{OUT}/X224.npy"):
    shutil.copy(CACHE[0], f"{OUT}/X224.npy")   # el mismo X224.npy, para que la salida sea el cache completo
print(json.dumps(stats, indent=1))

In [ ]:
# CELDA 7: verificaciones de fuga (test y val contra train)
m = pd.read_csv(f"{OUT}/meta.csv"); X2 = np.load(f"{OUT}/X224.npy", mmap_mode="r")
assert X2.shape == (len(m), 224, 224, 3) and (m.sample_id.values == meta.sample_id.values).all()
assert m.groupby("group_id").split.nunique().max() == 1, "un grupo quedo partido"
sp = m.split.values
tr = np.where(sp == "train")[0]
for parte in ["test", "val"]:
    ev = np.where(sp == parte)[0]
    mismo = cls[ev][:, None] == cls[tr][None, :]
    dt = np.abs(t[ev][:, None] - t[tr][None, :])
    sim = S[np.ix_(ev, tr)]
    print(f"--- {parte} ({len(ev)} imagenes) contra train")
    print("  con foto de train de la misma clase a <= 2 s :", int(((dt <= 2) & mismo).any(1).sum()))
    print("  con foto de train de la misma clase a <= %d s:" % T_RAFAGA, int(((dt <= T_RAFAGA) & mismo).any(1).sum()))
    print("  con gemela visual en train (ResNet50 >= %.2f):" % COS_MIN, int(((sim >= COS_MIN) & mismo).any(1).sum()))
    print("  con parecida en train (ResNet50 0.88-%.2f), limitacion:" % COS_MIN, int(((sim >= 0.88) & mismo).any(1).sum()))
    assert not ((dt <= T_RAFAGA) & mismo).any() and not ((sim >= COS_MIN) & mismo).any()
print("CACHE V2 OK | train/val/test:", m.split.value_counts().to_dict())
!ls -la /kaggle/working